# Lab 04 — Neural Networks for Classification and Regression

In this laboratory work, we will use **Keras** neural networks through the **SciKeras** interface and evaluate them using **scikit-learn cross-validation**.

We will solve three tasks independently:

1. **Binary classification** — Pima Indians Diabetes dataset
2. **Multiclass classification** — Iris dataset
3. **Regression** — Boston Housing dataset

The notebook is designed so that **Run All** works from top to bottom without manually selecting cells.

> Important methodological rule: all transformations that learn parameters from data (imputation, scaling, etc.) are fitted **inside each cross-validation training fold** using a `Pipeline`. This prevents data leakage.

## References

Primary documentation:

- Keras: https://keras.io/
- SciKeras: https://adriangb.com/scikeras/stable/
- scikit-learn cross-validation: https://scikit-learn.org/stable/modules/cross_validation.html
- scikit-learn Pipeline: https://scikit-learn.org/stable/modules/compose.html
- scikit-learn model evaluation: https://scikit-learn.org/stable/modules/model_evaluation.html

Additional tutorials:

- https://machinelearningmastery.com/binary-classification-tutorial-with-the-keras-deep-learning-library/
- https://machinelearningmastery.com/multi-class-classification-tutorial-keras-deep-learning-library/
- https://machinelearningmastery.com/regression-tutorial-keras-deep-learning-library-python/

## 1. Import libraries

In [ ]:
# Install SciKeras only if it is missing.
# In Jupyter, %pip is preferred because it installs into the active kernel environment.
try:
    import scikeras
except ImportError:
    %pip install -q scikeras

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from scikeras.wrappers import KerasClassifier, KerasRegressor

from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

# 2. Binary classification

We will predict whether a patient belongs to the diabetes-positive or diabetes-negative class.

The Pima dataset contains several variables where a value of `0` can represent a missing measurement rather than a physiologically meaningful value. We will convert these impossible zeros to `NaN`, but the **median used to fill missing values will be learned only from the training fold**.

In [ ]:
# Load the dataset
pima_url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"

pima_columns = [
    "Pregnancies",
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI",
    "DiabetesPedigreeFunction",
    "Age",
    "Outcome",
]

pima_df = pd.read_csv(pima_url, header=None, names=pima_columns)
pima_df.head()

In [ ]:
# Inspect data types, dimensions, class distribution, and missing values
print("Shape:", pima_df.shape)
print("\nData types:")
print(pima_df.dtypes)
print("\nClass distribution:")
print(pima_df["Outcome"].value_counts())

In [ ]:
# For these clinical variables, zero is treated as a missing measurement.
# Pregnancies and Outcome are NOT included because zero is a valid value there.
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
pima_df[zero_as_missing] = pima_df[zero_as_missing].replace(0, np.nan)

print("Missing values after zero -> NaN conversion:")
print(pima_df.isna().sum())

In [ ]:
# Split into predictors and target
X_binary = pima_df.drop(columns="Outcome")
y_binary = pima_df["Outcome"].astype(int)

print("X shape:", X_binary.shape)
print("y shape:", y_binary.shape)

In [ ]:
# Define the binary classification neural network.
# SciKeras provides information about the input data through the `meta` dictionary.
def build_binary_model(meta):
    n_features = meta["n_features_in_"]

    model = keras.Sequential([
        keras.Input(shape=(n_features,)),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

build_binary_model({"n_features_in_": X_binary.shape[1]}).summary()

In [ ]:
# IMPORTANT: imputation and scaling are inside the Pipeline.
# Therefore they are fitted separately in every training fold.
binary_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", KerasClassifier(
        model=build_binary_model,
        epochs=100,
        batch_size=16,
        verbose=0,
        random_state=RANDOM_STATE,
    )),
])

binary_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

binary_scores = cross_validate(
    binary_pipeline,
    X_binary,
    y_binary,
    cv=binary_cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "roc_auc": "roc_auc",
        "f1": "f1",
    },
)

binary_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Balanced accuracy", "ROC AUC", "F1"],
    "Mean": [
        binary_scores["test_accuracy"].mean(),
        binary_scores["test_balanced_accuracy"].mean(),
        binary_scores["test_roc_auc"].mean(),
        binary_scores["test_f1"].mean(),
    ],
    "SD": [
        binary_scores["test_accuracy"].std(),
        binary_scores["test_balanced_accuracy"].std(),
        binary_scores["test_roc_auc"].std(),
        binary_scores["test_f1"].std(),
    ],
})

binary_summary

### Questions

- Why is `StratifiedKFold` preferred over ordinary `KFold` for classification?
- Why must the imputer and scaler be inside the cross-validation pipeline?
- Why can balanced accuracy be more informative than ordinary accuracy when classes are imbalanced?

# 3. Multiclass classification

We will classify Iris flowers into three species. The target labels will be encoded as integers (`0`, `1`, `2`). One-hot encoding is not required because the neural network will use `sparse_categorical_crossentropy`.

In [ ]:
# Load the Iris dataset
iris_url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/iris.csv"
iris_columns = [
    "SepalLength",
    "SepalWidth",
    "PetalLength",
    "PetalWidth",
    "Species",
]

iris_df = pd.read_csv(iris_url, header=None, names=iris_columns)
iris_df.head()

In [ ]:
print("Shape:", iris_df.shape)
print("\nData types:")
print(iris_df.dtypes)
print("\nClass distribution:")
print(iris_df["Species"].value_counts())

In [ ]:
# Split predictors and target
X_multiclass = iris_df.drop(columns="Species")
y_multiclass_text = iris_df["Species"]

# Convert class names to integer labels
label_encoder = LabelEncoder()
y_multiclass = label_encoder.fit_transform(y_multiclass_text)

print("Classes:", list(label_encoder.classes_))
print("Encoded labels:", np.unique(y_multiclass))

In [ ]:
# Define the multiclass neural network.
def build_multiclass_model(meta):
    n_features = meta["n_features_in_"]
    n_classes = meta["n_classes_"]

    model = keras.Sequential([
        keras.Input(shape=(n_features,)),
        layers.Dense(8, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])

    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

build_multiclass_model({
    "n_features_in_": X_multiclass.shape[1],
    "n_classes_": len(np.unique(y_multiclass)),
}).summary()

In [ ]:
multiclass_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KerasClassifier(
        model=build_multiclass_model,
        epochs=100,
        batch_size=10,
        verbose=0,
        random_state=RANDOM_STATE,
    )),
])

multiclass_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

multiclass_scores = cross_validate(
    multiclass_pipeline,
    X_multiclass,
    y_multiclass,
    cv=multiclass_cv,
    scoring={
        "accuracy": "accuracy",
        "balanced_accuracy": "balanced_accuracy",
        "f1_macro": "f1_macro",
    },
)

multiclass_summary = pd.DataFrame({
    "Metric": ["Accuracy", "Balanced accuracy", "Macro F1"],
    "Mean": [
        multiclass_scores["test_accuracy"].mean(),
        multiclass_scores["test_balanced_accuracy"].mean(),
        multiclass_scores["test_f1_macro"].mean(),
    ],
    "SD": [
        multiclass_scores["test_accuracy"].std(),
        multiclass_scores["test_balanced_accuracy"].std(),
        multiclass_scores["test_f1_macro"].std(),
    ],
})

multiclass_summary

### Questions

- Why is the final layer a `softmax` layer with one output neuron per class?
- What is the difference between `categorical_crossentropy` and `sparse_categorical_crossentropy`?
- Why is macro F1 useful when evaluating multiclass models?

# 4. Regression

We will predict a continuous housing target. Unlike classification, regression does not use accuracy. We will report:

- **MAE** — mean absolute error
- **RMSE** — root mean squared error
- **R²** — coefficient of determination

The target variable is kept in its original units. Only the input features are standardized, and scaling is again performed inside each training fold.

> Note: the Boston Housing dataset is retained here to stay close to the original laboratory exercise. It is an older benchmark dataset and should be treated as a teaching example rather than a modern reference dataset.

In [ ]:
# Load the housing dataset
housing_url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/housing.data"
housing_df = pd.read_csv(housing_url, sep=r"\s+", header=None)

housing_df.head()

In [ ]:
print("Shape:", housing_df.shape)
print("\nData types:")
print(housing_df.dtypes)
print("\nMissing values:", housing_df.isna().sum().sum())

In [ ]:
# Last column is the regression target
X_regression = housing_df.iloc[:, :-1].astype(float)
y_regression = housing_df.iloc[:, -1].astype(float)

print("X shape:", X_regression.shape)
print("y shape:", y_regression.shape)

In [ ]:
# Define the regression neural network.
def build_regression_model(meta):
    n_features = meta["n_features_in_"]

    model = keras.Sequential([
        keras.Input(shape=(n_features,)),
        layers.Dense(8, activation="relu"),
        layers.Dense(1, activation="linear"),
    ])

    model.compile(
        optimizer="adam",
        loss="mean_squared_error",
    )
    return model

build_regression_model({"n_features_in_": X_regression.shape[1]}).summary()

In [ ]:
regression_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KerasRegressor(
        model=build_regression_model,
        epochs=100,
        batch_size=16,
        verbose=0,
        random_state=RANDOM_STATE,
    )),
])

regression_cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE,
)

regression_scores = cross_validate(
    regression_pipeline,
    X_regression,
    y_regression,
    cv=regression_cv,
    scoring={
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
        "r2": "r2",
    },
)

# scikit-learn returns errors as negative values because larger scorer values are considered better.
mae = -regression_scores["test_mae"]
rmse = -regression_scores["test_rmse"]
r2 = regression_scores["test_r2"]

regression_summary = pd.DataFrame({
    "Metric": ["MAE", "RMSE", "R²"],
    "Mean": [mae.mean(), rmse.mean(), r2.mean()],
    "SD": [mae.std(), rmse.std(), r2.std()],
})

regression_summary

### Questions

- Why is accuracy not appropriate for regression?
- Why are MAE and RMSE reported as positive values although scikit-learn uses negative error scorers internally?
- What does an R² value close to 1 mean?
- Why is the target variable not standardized in this example?

# 5. Summary

This laboratory demonstrated three neural-network problem types while following the same general workflow:

1. load and inspect the data;
2. define predictors (`X`) and target (`y`);
3. place learned preprocessing steps inside a `Pipeline`;
4. select a cross-validation strategy appropriate for the task;
5. define a task-specific neural-network output layer and loss function;
6. evaluate the model using appropriate metrics.

Key distinctions:

| Task | Output activation | Loss | CV | Example metrics |
|---|---|---|---|---|
| Binary classification | Sigmoid | Binary cross-entropy | StratifiedKFold | Accuracy, balanced accuracy, ROC AUC, F1 |
| Multiclass classification | Softmax | Sparse categorical cross-entropy | StratifiedKFold | Accuracy, balanced accuracy, macro F1 |
| Regression | Linear | Mean squared error | KFold | MAE, RMSE, R² |